In [2]:
import pandas as pd
import numpy as np

In [3]:
df_fe = pd.read_csv("../raw_data/features_enriched_labels.csv")

In [10]:
# =========================
# BASIC TYPE FIXES
# =========================
date_cols = ["founded_at", "first_funding_at", "last_funding_at"]
for col in date_cols:
    df_fe[col] = pd.to_datetime(df_fe[col], errors="coerce")

df_fe["funding_total_usd"] = pd.to_numeric(df_fe["funding_total_usd"], errors="coerce")

# =========================
# SNAPSHOT DATE
# =========================
# Fixed reference date for reproducibility
SNAPSHOT = pd.Timestamp("2015-01-01")


# =========================
# TIME SINCE LAST FUNDING
# =========================
df_fe["time_since_last_funding"] = (
    (SNAPSHOT - df_fe["last_funding_at"]).dt.days / 365.25
)

# Optional: negative values are data issues -> clip to 0
df_fe["time_since_last_funding"] = df_fe["time_since_last_funding"].clip(lower=0)

# =========================
# TARGET 1:
# acquired + operating vs closed
# =========================
df_fe["target_bin_acq_oper_vs_closed"] = df_fe["status_enriched"].map({
    "acquired": 1,
    "operating": 1,
    "closed": 0
})

# =========================
# TARGET 2:
# acquired vs closed
# operating + unknown -> NaN
# =========================
df_fe["target_bin_acq_vs_closed"] = df_fe["status_enriched"].map({
    "acquired": 1,
    "closed": 0
})

# =========================
# TARGET 3:
# multiclass
# unknown -> NaN
# =========================
df_fe["target_multiclass"] = df_fe["status_enriched"].map({
    "closed": 0,
    "operating": 1,
    "acquired": 2
})

# =========================
# PREP FOR TARGET 4:
# median funding by industry
# =========================
industry_median_funding = df_fe.groupby("industry_group")["funding_total_usd"].median()
df_fe["industry_median_funding"] = df_fe["industry_group"].map(industry_median_funding)

# =========================
# TARGET 4:
# refined target
# acquired = 1
# closed = 0
# operating > 4 years = 1
# operating <= 4 years:
#   funding above industry median = 1
#   else = 0
# unknown -> NaN
# =========================
def classify_status(row):
    status = row["status_enriched"]

    if status == "acquired":
        return 1

    elif status == "closed":
        return 0

    elif status == "operating":
        if pd.isna(row["time_since_last_funding"]) or pd.isna(row["industry_median_funding"]) or pd.isna(row["funding_total_usd"]):
            return np.nan

        if row["time_since_last_funding"] > 4:
            return 1
        else:
            return 1 if row["funding_total_usd"] > row["industry_median_funding"] else 0

    return np.nan

df_fe["target_bin_refined"] = df_fe.apply(classify_status, axis=1)

# =========================
# FINAL CHECKS
# =========================
target_cols = [
    "target_bin_acq_oper_vs_closed",
    "target_bin_acq_vs_closed",
    "target_multiclass",
    "target_bin_refined"
]

for col in target_cols:
    print(f"\n{col}")
    print(df_fe[col].value_counts(dropna=False, normalize=True))

# Additional sanity checks
print("\nMissing values in key engineered columns:")
print(
    df_fe[
        [
            "time_since_last_funding",
            "industry_median_funding",
            "funding_total_usd"
        ]
    ].isna().mean().sort_values(ascending=False)
)


target_bin_acq_oper_vs_closed
target_bin_acq_oper_vs_closed
1.0    0.725402
0.0    0.248653
NaN    0.025945
Name: proportion, dtype: float64

target_bin_acq_vs_closed
target_bin_acq_vs_closed
NaN    0.687803
0.0    0.248653
1.0    0.063544
Name: proportion, dtype: float64

target_multiclass
target_multiclass
1.0    0.661858
0.0    0.248653
2.0    0.063544
NaN    0.025945
Name: proportion, dtype: float64

target_bin_refined
target_bin_refined
1.0    0.575998
0.0    0.398057
NaN    0.025945
Name: proportion, dtype: float64

Missing values in key engineered columns:
years_operating            0.0
time_since_last_funding    0.0
industry_median_funding    0.0
funding_total_usd          0.0
dtype: float64


In [8]:
df_fe.columns

Index(['permalink', 'category_list', 'market', 'funding_total_usd',
       'country_code', 'state_code', 'region', 'funding_rounds', 'founded_at',
       'first_funding_at', 'last_funding_at', 'seed', 'venture',
       'debt_financing', 'angel', 'grant', 'private_equity', 'round_A',
       'round_B', 'round_C', 'round_D', 'round_E', 'status_enriched',
       'avg_raised_per_round', 'age_first_funding_days', 'has_multiple_rounds',
       'funding_span_days', 'avg_years_between_rounds', 'region_group',
       'market_clean', 'industry_group', 'time_since_last_funding',
       'target_bin_acq_oper_vs_closed', 'target_bin_acq_vs_closed',
       'target_multiclass', 'industry_median_funding', 'years_operating',
       'target_bin_refined'],
      dtype='object')

In [12]:
df_fe.to_csv("features_enriched_with_targets.csv", index=False)
